# Load the packages

In [1]:
import numpy as np
import pandas as pd
import cv2
import datetime
import glob
import time
import os

from detect import detect_objects, logData, frame_bb, save
from test_depth_est import load_detections_from_csv, compute_disparity, process_frame

# Load the data

In [2]:
seq = 3

currentPath = "C:/Users/shaia/Documents/Opgaveregning/AutoSys/4. semester/Perception for autonome systemer/Eksamprojekt/"
dataPathLeft = currentPath + f"34759_final_project_raw/seq_0{seq}/image_02/"
dataPathRight = currentPath + f"34759_final_project_raw/seq_0{seq}/image_03/"
dataPNGLeft = dataPathLeft + "data/"
dataPNGRight = dataPathRight + "data/"
dataTime = dataPathRight + "timestamps.txt"

dataLeft = glob.glob(f"{dataPNGLeft}*.png")
dataRight = glob.glob(f"{dataPNGRight}*.png")

timeStamp = open(dataTime)
timeData = timeStamp.readlines()
timeStamp.close()
time1 = datetime.datetime.strptime(timeData[0][:-4], "%Y-%m-%d %H:%M:%S.%f")
time2 = datetime.datetime.strptime(timeData[1][:-4], "%Y-%m-%d %H:%M:%S.%f")
deltaT = (time2 - time1).total_seconds()

# Rectify the raw images

In [3]:
leftImg = []

for i in range(len(dataLeft)-1):
    imgLeft = cv2.imread(dataLeft[i])
    imgRight = cv2.imread(dataRight[i])
    _, _, left_rect = compute_disparity(imgLeft, imgRight)
    leftImg.append(left_rect)

# Make the Kalman filter

In [4]:
states = 4

x0 = np.zeros((3 * states, 1))
u = np.zeros(x0.shape)
P0car = np.diag([100000, 100, 0.1] * states)
P0ped = np.diag([100000, 0.01, 0.000001] * states)
P0cyc = np.diag([100000, 0.01, 0.0001] * states)
R = 0.0001 * np.eye(states)

F = np.eye(x0.shape[0])
for i in range(F.shape[0]):
    try:
        F[i, i + 1] = deltaT if i % 3 != 2 and i < F.shape[0] - 1 else 0
        F[i, i + 2] = 0.5 * deltaT ** 2 if i % 3 == 0 and i < F.shape[0] - 2 else 0
    except:
        pass

H = np.zeros((R.shape[0], x0.shape[0]))
for i in range(H.shape[0]):
    H[i, i * 3] = 1
    
kalman0 = {"x": x0, "u": u, "Pcar": P0car, "Pped": P0ped, "Pcyc": P0cyc, "F": F, "H": H, "R": R}

# Find the bounding box and get the dataframe

In [5]:
bbox = detect_objects(leftImg)
df = logData(leftImg, bbox, kalman0)

video = []
for i in range(max(df["frame"])):
    img = leftImg[i]
    
    if i not in df["frame"]:
        video.append(img)
        continue
    
    subData = df.loc[df["frame"] == i]
    coords = np.array([subData["bbox left"], subData["bbox top"], subData["bbox right"], subData["bbox bottom"]]).T
    types = list(subData["type"])
    trackID = list(subData["track id"])

    imgFrame = frame_bb(img, coords, types, trackID)
    video.append(imgFrame)


0: 256x640 2 cars, 1 pedestrian, 90.4ms
Speed: 2.4ms preprocess, 90.4ms inference, 2.2ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 2 cars, 1 pedestrian, 46.6ms
Speed: 3.2ms preprocess, 46.6ms inference, 2.8ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 3 cars, 1 pedestrian, 49.9ms
Speed: 1.9ms preprocess, 49.9ms inference, 1.6ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 3 cars, 1 pedestrian, 44.1ms
Speed: 1.9ms preprocess, 44.1ms inference, 1.7ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 3 cars, 1 pedestrian, 49.4ms
Speed: 1.9ms preprocess, 49.4ms inference, 1.8ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 3 cars, 1 pedestrian, 1 cyclist, 45.2ms
Speed: 1.9ms preprocess, 45.2ms inference, 1.8ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640 3 cars, 1 pedestrian, 2 cyclists, 45.5ms
Speed: 2.6ms preprocess, 45.5ms inference, 2.1ms postprocess per image at shape (1, 3, 256, 640)

0: 256x640

# Save the image and CSV file

In [6]:
save(video, df, 1 / deltaT, f"ResultSeq{seq}")

# Estimating the depth

In [7]:
left_detections = load_detections_from_csv(currentPath + f"ResultSeq{seq}.csv")

num_frames = min(len(dataLeft), len(dataRight))
has_saved_disparity = False

for i in range(num_frames):
    start_time = time.time()

    frameL = cv2.imread(dataLeft[i])
    frameR = cv2.imread(dataRight[i])

    if frameL is None or frameR is None: continue
    annotated_img, disparity_map = process_frame(i, frameL, frameR, left_detections)

    if not has_saved_disparity:
        disp_vis = cv2.normalize(disparity_map, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        disp_color = cv2.applyColorMap(disp_vis, cv2.COLORMAP_JET)
        cv2.imwrite("single_disparity_sample.png", disp_color)
        has_saved_disparity = True
        print("Saved 'single_disparity_sample.png'")

    elapsed = time.time() - start_time
    fps = 1.0 / elapsed if elapsed > 0 else 0
    cv2.putText(annotated_img, f"FPS: {fps:.1f}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    cv2.imshow("Stereo CSV Depth", annotated_img)
        
    disp_live = cv2.normalize(disparity_map, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    cv2.imshow("Disparity Live", disp_live)
        
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()

Saved 'single_disparity_sample.png'
